# Especialização em Análise de Dados no Futebol — 2026
## FOOTURE ACADEMY — DESAFIO FINAL
Competição: Campeonato Brasileiro Série A \
Temporada: 2025 \
Posição Analisada: Zagueiro \
Data: Janeiro/2026 \
Responsável: Daniel Alves Pinho

> #### Objetivos:
> 1. Criação de uma métrica autoral para avaliação de jogadores.
> 2. Análise de Mercado e Lista de Reforços (Scout)

####**Parte 2: Análise de Mercado e Lista de Reforços (Scout)**
*2.1. Importação de bilbiotecas*
> i. Vamos importar as bibliotecas do Python cujas funções leem planilhas, bem como executam ações de filtragem e limpeza nos dados. A saber, o Pandas. Já a bilbioteca Scikit-learn contém o módulo de pré-procesamento, cuja função MinMaxScaler será utilizada para balancear nossos dados. \
> ii. Iremos importar, um pouco mais afrente no código, a biblioteca Matplotlib, para gerar gráficos personalizados com os dados obtidos

In [83]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

*2.2. Leitura de Planilhas & Extração dos Dados*
> i. Utilizamos a planilha cedida pela Footure Academy disponilizada pela WyScout. \
> ii. Lá encontramos muitos dados faltando, e fora de formatação. Por isso é preciso passar algumas filtragens e funções.

In [84]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/projeto_final_footure/data/CB.BR2025.csv')
df.columns = df.columns.str.strip()

In [85]:
renomear_colunas = {
    "Equipa": "Equipe",
    "Interseções/90": "Interceptações/90",
    "Interceções/90": "Interceptações/90",
    "Interceções ajust. à posse": "Interceptações ajustadas à posse",
    "Duelos aérios/90...25": "Duelos aéreos/90",
    "Ações defensivas com êxito/90": "Ações defensivas bem-sucedidas/90",
    "Cortes de carrinho ajust. à posse": "Carrinhos ajustados à posse",
    "Remates intercetados/90": "Chutes bloqueados/90",
    "Duelos defensivos ganhos, %": "Taxa de sucesso em duelos defensivos",
    "Duelos aéreos ganhos, %": "Taxa de sucesso em duelos aéreos",
    "Cartões amarelos/90": "Cartões amarelos por 90",
    "Cartões vermelhos/90": "Cartões vermelhos por 90"
}

df = df.rename(columns=renomear_colunas)

In [86]:
metricas_defensivas = [
    "Ações defensivas bem-sucedidas/90",
    "Interceptações/90",
    "Interceptações ajustadas à posse",
    "Chutes bloqueados/90",
    "Cortes/90",
    "Carrinhos ajustados à posse",
    "Duelos defensivos/90",
    "Taxa de sucesso em duelos defensivos",
    "Duelos aéreos/90",
    "Taxa de sucesso em duelos aéreos",
    "Faltas/90",
    "Cartões amarelos por 90",
    "Cartões vermelhos por 90"
]

*2.3. Limpeza dos Dados*
> i. Renomeamos nomes que estão em Português de Portugal, para termos mais clareza semântica, bem como corrigimos alguns erros ortográficos e de padronização. \

In [87]:
df_zag = df[
    df["Posição"].str.contains(r"\bCB\b|\bRCB\b|\bLCB\b", case=False, na=False)
].copy()

In [88]:
metricas_existentes = [m for m in metricas_defensivas if m in df_zag.columns]

for col in metricas_existentes:
    df_zag[col] = pd.to_numeric(df_zag[col], errors="coerce")


In [89]:
df_zag = df_zag[df_zag["Minutos jogados:"] >= 900 & (df_zag["Idade"] <= 29 & (df_zag["Equipe"] != "Vitória"))]

In [90]:
df_zag = df_zag.dropna(
    subset=[
        "Ações defensivas bem-sucedidas/90",
        "Duelos defensivos/90",
        "Duelos aéreos/90"
    ]
)

*2.4. Filtros*
> i. Fazemos uma filtragem por posição. No caso do Lucas Halter, um zagueiro que atua pela esquerda ou direita. Logo, na tabela CB, RCB e LCB. \
> ii. Padronizamos as métricas que iremos utilizar para valores numéricos \
> iii. Utilizamos uma proporção de minutagem de pelo menos 10 partidas (900 minutos), além de não ser um jogador do Vitória (vide que estamos querendo contratar um novo atleta), e sua idade ser abaixo de 29, o que é coerente para posição de zagueiro e o nível de Halter.\
> iv. Definimos um subset que será o alicerce da nossa métrica. Esta linha de código permite excluir o atleta da lista caso ele tenha valores ausentes em pelo menos uma dessas valências

In [91]:
scaler = MinMaxScaler()
df_zag_norm = df_zag.copy()

df_zag_norm[metricas_existentes] = scaler.fit_transform(
    df_zag_norm[metricas_existentes]
)

*2.5. Normalização de Dados*
>i. O MinMaxScaler() permite a normalização das métricas em escala comum (0-1), garantindo comparabilidade entre atletas e evitando que diferenças de magnitude numérica distorçam nosso modelo.

In [92]:
df_zag_norm["Score Defensivo"] = (
    df_zag_norm["Ações defensivas bem-sucedidas/90"] * 0.20 +
    df_zag_norm["Taxa de sucesso em duelos defensivos"] * 0.15 +
    df_zag_norm["Taxa de sucesso em duelos aéreos"] * 0.15 +
    df_zag_norm["Interceptações/90"] * 0.15 +
    df_zag_norm["Cortes/90"] * 0.10 +
    df_zag_norm["Chutes bloqueados/90"] * 0.10 -
    df_zag_norm["Faltas/90"] * 0.10 -
    df_zag_norm["Cartões amarelos por 90"] * 0.03 -
    df_zag_norm["Cartões vermelhos por 90"] * 0.02
).round(2)


*2.6. Definição de Métrica*
>i. Com base nos critérios propostos no Relatório, aplicamos os seguintes pesos, de soma 1, para criação de nossas métricas.

In [93]:
ranking = df_zag_norm.sort_values("Score Defensivo", ascending=False)


>ii. Cria-se uma forma de ranquear os atletas de acordo com a métrica normalizada.

In [94]:
orcamento_total = 1000000
orcamento_titulares = 700000
orcamento_elenco = 300000


>iii. Geramos variáveis distribuindo o orçamento do clube.

In [95]:
titulares = []
custo_titulares = 0

for _, row in ranking.iterrows():
    if custo_titulares + row["Valor de mercado"] <= orcamento_titulares:
        titulares.append(row)
        custo_titulares += row["Valor de mercado"]
    if len(titulares) == 3:
        break

titulares_df = pd.DataFrame(titulares)

>iv. Filtramos por possíveis titulares dentro do orçamento cruzando com as métricas defensivas.



In [96]:
ranking_restante = ranking.drop(titulares_df.index)

elenco = []
custo_elenco = 0

for _, row in ranking_restante.iterrows():
    if custo_elenco + row["Valor de mercado"] <= orcamento_elenco:
        elenco.append(row)
        custo_elenco += row["Valor de mercado"]
    if len(elenco) == 3:
        break

elenco_df = pd.DataFrame(elenco)

>v. Entedemos que por não ser um possível titular, todo candidato restante seria um atleta com potencial de compor elenco, já com as métricas cruzadas.

In [97]:
parceiros = ranking[
    (ranking["Idade"] >= 22) &
    (ranking["Valor de mercado"] <= 600000) &
    (ranking["Faltas/90"] <= ranking["Faltas/90"].quantile(0.60)) &
    (ranking["Cartões amarelos por 90"] <= ranking["Cartões amarelos por 90"].quantile(0.60))
].copy()

parceiros["Score Compatibilidade Halter"] = (
    parceiros["Interceptações/90"] * 0.35 +
    parceiros["Interceptações ajustadas à posse"] * 0.20 +
    parceiros["Ações defensivas bem-sucedidas/90"] * 0.20 +
    parceiros["Chutes bloqueados/90"] * 0.15 -
    parceiros["Faltas/90"] * 0.10
).round(2)

ranking_parceiro_halter = parceiros.sort_values(
    "Score Compatibilidade Halter",
    ascending=False
)

parceiro_ideal = ranking_parceiro_halter.head(1)[
    ["Jogador", "Equipe", "Idade", "Valor de mercado",
     "Score Defensivo", "Score Compatibilidade Halter"]
]

>vi. Ao final, usamos as métricas do próprio Lucas Halter e cruzamos com o resultado de Jogadores Titulares em Potencial, bem como Opções de Elenco, criando um Score de compatibilidade com o atleta.

In [98]:
print("TITULARES IMEDIATOS")
display(titulares_df[["Jogador", "Equipe", "Idade", "Valor de mercado", "Score Defensivo"]])
print(f"Custo Titulares: R$ {custo_titulares:,.0f}")

print("\n OPÇÕES PARA ELENCO")
display(elenco_df[["Jogador", "Equipe", "Idade", "Valor de mercado", "Score Defensivo"]])
print(f"Custo Elenco: R$ {custo_elenco:,.0f}")

print("\n PARCEIRO IDEAL PARA LUCAS HALTER (VITÓRIA)")
display(parceiro_ideal)

print("\n ORÇAMENTO TOTAL UTILIZADO")
print(f"R$ {custo_titulares + custo_elenco:,.0f}")

TITULARES IMEDIATOS


,Jogador,Equipe,Idade,Valor de mercado,Score Defensivo
129,Viery,Grêmio U20,20,0.0,0.53
127,Cleiton,Flamengo,22,400000.0,0.46
121,Gabriel Bahia,Botafogo,27,0.0,0.38


Custo Titulares: R$ 400,000

 OPÇÕES PARA ELENCO


,Jogador,Equipe,Idade,Valor de mercado,Score Defensivo
102,Gabriel,Mirassol,22,0.0,0.34
48,W. Ángel,Juventude,32,250000.0,0.32
53,Abner,Juventude,21,0.0,0.31


Custo Elenco: R$ 250,000

 PARCEIRO IDEAL PARA LUCAS HALTER (VITÓRIA)


,Jogador,Equipe,Idade,Valor de mercado,Score Defensivo,Score Compatibilidade Halter
127,Cleiton,Flamengo,22,400000.0,0.46,0.49



 ORÇAMENTO TOTAL UTILIZADO
R$ 650,000
